# 03 — Exact Name Blocking Tournament
**Amazon ML Challenge 2026: Business Entity Resolution**

Compares raw clean name (A1), suffix-stripped core name (A2), and token-sorted name (A3).


### 1. Environment & Setup


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Adjust sys.path to root directory
if '..' not in sys.path:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))
if '.' not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.business_entity_resolution.blocking import (
    build_or_load_benchmark,
    evaluate_candidate_pairs,
    format_evaluation_summary,
    CandidateUnionEngine
)


### 2. Load Benchmark Dataset


In [ ]:
s1, s2, s3, gt = build_or_load_benchmark(n_s1=5000, n_distractors=100000, random_seed=42)
target_pool = pd.concat([s2, s3], ignore_index=True)


### 3. Run Multi-Representation Name Blocker


In [ ]:
from src.business_entity_resolution.blocking import NameBlockConfig, ExactNameBlocker

for name_mode, kwargs in [("Raw Clean", {"use_raw_clean": True, "use_core_name": False, "use_sorted_tokens": False}),
                          ("Core Name (Suffix Stripped)", {"use_raw_clean": False, "use_core_name": True, "use_sorted_tokens": False}),
                          ("Token Sorted", {"use_raw_clean": False, "use_core_name": False, "use_sorted_tokens": True}),
                          ("All Representations", {"use_raw_clean": True, "use_core_name": True, "use_sorted_tokens": True})]:
    cfg = NameBlockConfig(**kwargs)
    blk = ExactNameBlocker(name=name_mode, config=cfg)
    blk.fit(target_pool)
    cands = blk.block(s1)
    m = evaluate_candidate_pairs(cands, gt, len(target_pool))
    print(f"[{name_mode}] Pair Recall: {m['pair_recall']*100:.2f}% | Avg Cands: {m['avg_candidates']:.2f}")
